# Demo 1 — Historical Batch Ingestion

This notebook ingests the three Binance daily OHLCV CSV files from the external Volume into a Bronze Delta table.

It demonstrates:
- reading multiple headerless CSV files
- applying an explicit schema
- deriving the crypto symbol from the file name
- converting Binance microsecond timestamps
- adding ingestion metadata
- validating the source data
- using Delta `MERGE` for idempotent reruns

> Rerunning this notebook will not create duplicate rows because the target table is merged using `symbol + open_time`.

## 1. Load shared configuration

All paths and table names come from `config/00_config`.

In [0]:
%run ../config/00_config

## 2. Import required Spark functions and data types

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)

## 3. Validate source files

The source folder should contain exactly these three CSV files:

- `BTCUSDT-1d-2026-01.csv`
- `ETHUSDT-1d-2026-01.csv`
- `SOLUSDT-1d-2026-01.csv`

The `_READY` file is ignored.

In [0]:
expected_files = {
    "BTCUSDT-1d-2026-01.csv",
    "ETHUSDT-1d-2026-01.csv",
    "SOLUSDT-1d-2026-01.csv",
}

source_files = {
    file_info.name
    for file_info in dbutils.fs.ls(historical_raw_path)
    if file_info.name.lower().endswith(".csv")
}

missing_files = expected_files - source_files

if missing_files:
    raise FileNotFoundError(
        f"Missing required historical CSV files: {sorted(missing_files)}"
    )

print("Historical source files found:")
for file_name in sorted(source_files):
    print(f"- {file_name}")

## 4. Define the Binance Kline schema

The downloaded Binance files do not contain a header row, so column names must be supplied explicitly.

Each row contains 12 fields.

In [0]:
kline_schema = StructType([
    StructField("open_time_raw", LongType(), False),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close_time_raw", LongType(), False),
    StructField("quote_asset_volume", DoubleType(), True),
    StructField("number_of_trades", LongType(), True),
    StructField("taker_buy_base_asset_volume", DoubleType(), True),
    StructField("taker_buy_quote_asset_volume", DoubleType(), True),
    StructField("ignore", StringType(), True),
])

## 5. Read the historical CSV files

The files use a fixed trusted schema, so this batch load uses `FAILFAST`.

This is different from the future streaming notebook, where schema evolution and rescued data will be demonstrated.

In [0]:
raw_df = (
    spark.read
    .format("csv")
    .option("header", "false")
    .option("mode", "FAILFAST")
    .schema(kline_schema)
    .load(f"{historical_raw_path}/*.csv")
)

print(f"Raw rows read: {raw_df.count()}")
display(raw_df.limit(10))

## 6. Transform raw rows into Bronze records

The transformation adds:
- `symbol` from the source file name
- proper timestamp columns
- source file metadata
- source-system metadata
- ingestion timestamp
- deterministic record key

Binance archive timestamps in these files are stored in microseconds, so they are divided by `1,000,000` before conversion.

In [0]:
bronze_source_df = (
    raw_df
    .withColumn("source_file_name", F.col("_metadata.file_name"))
    .withColumn("source_file_path", F.col("_metadata.file_path"))
    .withColumn(
        "symbol",
        F.regexp_extract(
            F.col("source_file_name"),
            r"^([A-Z0-9]+)-1d-",
            1,
        ),
    )
    .withColumn(
        "open_time",
        F.to_timestamp(
            F.from_unixtime(
                (F.col("open_time_raw") / F.lit(1_000_000)).cast("long")
            )
        ),
    )
    .withColumn(
        "close_time",
        F.to_timestamp(
            F.from_unixtime(
                (F.col("close_time_raw") / F.lit(1_000_000)).cast("long")
            )
        ),
    )
    .withColumn("interval", F.lit(historical_interval))
    .withColumn("source_system", F.lit(source_system_historical))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn(
        "record_id",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("symbol"),
                F.col("open_time_raw").cast("string"),
            ),
            256,
        ),
    )
    .select(
        "record_id",
        "symbol",
        "interval",
        "open_time",
        "close_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume",
        "open_time_raw",
        "close_time_raw",
        "source_system",
        "source_file_name",
        "source_file_path",
        "ingested_at",
    )
)

display(bronze_source_df.orderBy("symbol", "open_time"))

## 7. Validate transformed data

The validation checks:
- symbols are present
- timestamps were converted successfully
- OHLC prices are not null
- every symbol has data
- duplicate business keys do not exist in the source

In [0]:
invalid_rows_df = bronze_source_df.filter(
    F.col("symbol").isNull()
    | (F.col("symbol") == "")
    | F.col("open_time").isNull()
    | F.col("close_time").isNull()
    | F.col("open").isNull()
    | F.col("high").isNull()
    | F.col("low").isNull()
    | F.col("close").isNull()
)

invalid_row_count = invalid_rows_df.count()

if invalid_row_count > 0:
    display(invalid_rows_df)
    raise ValueError(
        f"Historical source contains {invalid_row_count} invalid rows."
    )

duplicate_keys_df = (
    bronze_source_df
    .groupBy("symbol", "open_time")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_count = duplicate_keys_df.count()

if duplicate_key_count > 0:
    display(duplicate_keys_df)
    raise ValueError(
        f"Historical source contains {duplicate_key_count} duplicate business keys."
    )

symbol_summary_df = (
    bronze_source_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("row_count"),
        F.min("open_time").alias("first_open_time"),
        F.max("open_time").alias("last_open_time"),
    )
    .orderBy("symbol")
)

display(symbol_summary_df)
print("Source validation passed.")

## 8. Create the Bronze table if it does not exist

The first run creates an empty Delta table using the transformed DataFrame schema.

In [0]:
if not spark.catalog.tableExists(historical_bronze_table):
    (
        bronze_source_df
        .limit(0)
        .write
        .format("delta")
        .mode("error")
        .saveAsTable(historical_bronze_table)
    )
    print(f"Created Bronze table: {historical_bronze_table}")
else:
    print(f"Bronze table already exists: {historical_bronze_table}")

## 9. Merge into the Bronze table

The business key is:

`symbol + open_time`

Existing rows are updated and new rows are inserted. This makes the ingestion idempotent.

In [0]:
bronze_source_df.createOrReplaceTempView("demo1_historical_source")

spark.sql(
    f"""
    MERGE INTO {historical_bronze_table} AS target
    USING demo1_historical_source AS source
      ON target.symbol = source.symbol
     AND target.open_time = source.open_time

    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """
)

print(f"Merge completed: {historical_bronze_table}")

## 10. Validate the Bronze table

The final checks confirm the row count, symbol coverage, and date range after the merge.

In [0]:
bronze_table_df = spark.table(historical_bronze_table)

bronze_summary_df = (
    bronze_table_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("row_count"),
        F.min("open_time").alias("first_open_time"),
        F.max("open_time").alias("last_open_time"),
        F.min("low").alias("minimum_low"),
        F.max("high").alias("maximum_high"),
    )
    .orderBy("symbol")
)

display(bronze_summary_df)

total_rows = bronze_table_df.count()
distinct_keys = (
    bronze_table_df
    .select("symbol", "open_time")
    .distinct()
    .count()
)

if total_rows != distinct_keys:
    raise ValueError(
        "Bronze table contains duplicate symbol + open_time keys."
    )

print("Historical batch ingestion completed successfully.")
print(f"Target table: {historical_bronze_table}")
print(f"Total rows: {total_rows}")
print(f"Distinct business keys: {distinct_keys}")

## 11. Rerun test

Run this entire notebook again.

The total row count should remain unchanged because `MERGE` updates existing rows instead of appending duplicates.

Expected result for January 2026:

- approximately 31 rows per symbol
- approximately 93 rows in total

In [0]:
%sql
select * from demo1_historical_source